# nb_gold_stream_charging_progress

**Source :** Silver Delta stream — `sl_vehicle_battery_live/`  
**Sink   :** `abfss://gold@evdatalakedev2.dfs.core.windows.net/mart_charging_progress_live/`  
**Grain  :** One row per `vehicle_id + station_id + 5-minute tumbling window`  
**Trigger:** Every 120 seconds via `foreachBatch`

Aggregates per vehicle per 5-minute window:
- Battery % trajectory (min → max → avg)
- Charging rate and temperature averages
- Overtemperature flag (any reading > 45°C in window)
- Estimated minutes to full at window close
- Event count (data density indicator)

MERGEs on `vehicle_id + station_id + window_start` so re-processing  
a window (partial data arriving late) updates rather than duplicates.

In [ ]:
# ── Cell 1: secrets ───────────────────────────────────────────────────────────
SP_CLIENT_ID  = dbutils.secrets.get(scope="ev-kv-scope", key="sp-client-id")
SP_CLIENT_SEC = dbutils.secrets.get(scope="ev-kv-scope", key="sp-client-secret")
SP_TENANT_ID  = dbutils.secrets.get(scope="ev-kv-scope", key="sp-tenant-id")
ADLS_ACCOUNT  = dbutils.secrets.get(scope="ev-kv-scope", key="adls-account-name")

In [ ]:
# ── Cell 2: ADLS OAuth config ─────────────────────────────────────────────────
spark.conf.set(f"fs.azure.account.auth.type.{ADLS_ACCOUNT}.dfs.core.windows.net",       "OAuth")
spark.conf.set(f"fs.azure.account.oauth.provider.type.{ADLS_ACCOUNT}.dfs.core.windows.net",
               "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set(f"fs.azure.account.oauth2.client.id.{ADLS_ACCOUNT}.dfs.core.windows.net",     SP_CLIENT_ID)
spark.conf.set(f"fs.azure.account.oauth2.client.secret.{ADLS_ACCOUNT}.dfs.core.windows.net", SP_CLIENT_SEC)
spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{ADLS_ACCOUNT}.dfs.core.windows.net",
               f"https://login.microsoftonline.com/{SP_TENANT_ID}/oauth2/token")
print("ADLS OAuth configured.")

In [ ]:
# ── Cell 3: paths ─────────────────────────────────────────────────────────────
SILVER_PATH     = f"abfss://silver@{ADLS_ACCOUNT}.dfs.core.windows.net/sl_vehicle_battery_live/"
GOLD_PATH       = f"abfss://gold@{ADLS_ACCOUNT}.dfs.core.windows.net/mart_charging_progress_live/"
CHECKPOINT_PATH = f"abfss://gold@{ADLS_ACCOUNT}.dfs.core.windows.net/_checkpoints/charging-progress-live/"

print(f"Source (Silver) : {SILVER_PATH}")
print(f"Sink   (Gold)   : {GOLD_PATH}")

In [ ]:
# ── Cell 4: Gold mart build function ──────────────────────────────────────────
from pyspark.sql import functions as F
from delta.tables import DeltaTable
import time

WINDOW_DURATION = "5 minutes"   # tumbling window size

def build_gold_mart(batch_df, batch_id):
    print(f"\n[Gold Batch {batch_id}] input rows = {batch_df.count()}")
    if batch_df.rdd.isEmpty():
        print(f"[Gold Batch {batch_id}] empty — skipping.")
        return

    # Parse event_ts string → timestamp (Silver stores it as string from producer)
    df = batch_df.withColumn(
        "event_ts_ts",
        F.to_timestamp(F.col("event_ts"))
    )

    # Tumbling 5-minute window aggregation
    mart_df = (
        df
        .groupBy(
            F.col("vehicle_id"),
            F.col("station_id"),
            F.window(F.col("event_ts_ts"), WINDOW_DURATION),
        )
        .agg(
            F.min("battery_pct")                    .alias("battery_pct_start"),
            F.max("battery_pct")                    .alias("battery_pct_end"),
            F.round(F.avg("battery_pct"), 2)        .alias("battery_pct_avg"),
            F.round(F.avg("charging_rate_kw"), 2)   .alias("avg_charging_rate_kw"),
            F.round(F.avg("battery_temp_c"), 2)     .alias("avg_temp_c"),
            F.round(F.max("battery_temp_c"), 2)     .alias("max_temp_c"),
            # Overtemperature: any reading above 45°C in this window
            F.max(F.when(F.col("battery_temp_c") > 45.0, 1).otherwise(0)).alias("overtemp_flag"),
            # Closest ETA at the end of the window (minimum = most up-to-date)
            F.min("estimated_minutes_to_full")      .alias("eta_minutes_to_full"),
            F.first("session_id")                   .alias("session_id"),
            F.first("charger_id")                   .alias("charger_id"),
            F.count("*")                            .alias("event_count"),
        )
        .select(
            F.col("vehicle_id"),
            F.col("station_id"),
            F.col("window.start").alias("window_start"),
            F.col("window.end")  .alias("window_end"),
            F.col("session_id"),
            F.col("charger_id"),
            F.col("battery_pct_start"),
            F.col("battery_pct_end"),
            F.col("battery_pct_avg"),
            F.col("avg_charging_rate_kw"),
            F.col("avg_temp_c"),
            F.col("max_temp_c"),
            F.col("overtemp_flag").cast("boolean"),
            F.col("eta_minutes_to_full"),
            F.col("event_count"),
            F.current_timestamp().alias("gold_updated_at"),
            F.lit(str(batch_id)).alias("gold_batch_id"),
        )
    )

    # MERGE into Gold on (vehicle_id, station_id, window_start)
    # Handles partial windows arriving across multiple Silver batches
    if DeltaTable.isDeltaTable(spark, GOLD_PATH):
        tgt = DeltaTable.forPath(spark, GOLD_PATH)
        (
            tgt.alias("t")
               .merge(
                   mart_df.alias("s"),
                   "t.vehicle_id = s.vehicle_id "
                   "AND t.station_id = s.station_id "
                   "AND t.window_start = s.window_start"
               )
               .whenMatchedUpdateAll()
               .whenNotMatchedInsertAll()
               .execute()
        )
    else:
        mart_df.write.format("delta").mode("overwrite").partitionBy("vehicle_id").save(GOLD_PATH)

    print(f"[Gold Batch {batch_id}] MERGE done. Window rows = {mart_df.count()}")

In [ ]:
# ── Cell 5: read Silver stream + trigger foreachBatch ─────────────────────────
silver_stream = (
    spark.readStream
    .format("delta")
    .load(SILVER_PATH)
)

gold_query = (
    silver_stream
    .writeStream
    .foreachBatch(build_gold_mart)
    .option("checkpointLocation", CHECKPOINT_PATH)
    .trigger(processingTime="120 seconds")
    .start()
)

print(f"Gold stream started. Query ID: {gold_query.id}")

In [ ]:
# ── Cell 6: keep-alive monitor loop ──────────────────────────────────────────
import time

while gold_query.isActive:
    prog = gold_query.lastProgress
    if prog:
        print(f"[Gold] batch={prog.get('batchId','?')}  inputRows={prog.get('numInputRows',0)}")
    time.sleep(120)

gold_query.awaitTermination()

In [ ]:
# ── Cell 7: live dashboard query (run manually) ───────────────────────────────
spark.sql(f"""
    SELECT
        vehicle_id,
        station_id,
        window_start,
        window_end,
        battery_pct_start,
        battery_pct_end,
        ROUND(battery_pct_end - battery_pct_start, 2)  AS pct_gained,
        avg_charging_rate_kw,
        avg_temp_c,
        max_temp_c,
        overtemp_flag,
        eta_minutes_to_full,
        event_count,
        gold_updated_at
    FROM delta.`{GOLD_PATH}`
    ORDER BY window_start DESC, vehicle_id
    LIMIT 50
""").display()

In [ ]:
# ── Cell 8: overtemperature alert summary (run manually) ─────────────────────
spark.sql(f"""
    SELECT
        vehicle_id,
        station_id,
        COUNT(*) FILTER (WHERE overtemp_flag = true)  AS overtemp_windows,
        MAX(max_temp_c)                               AS peak_temp_c,
        ROUND(AVG(avg_temp_c), 2)                     AS overall_avg_temp_c
    FROM delta.`{GOLD_PATH}`
    GROUP BY vehicle_id, station_id
    ORDER BY overtemp_windows DESC, peak_temp_c DESC
""").display()